In [ ]:
import os
import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers.cache_utils import DynamicCache
from sentence_transformers import SentenceTransformer, util
from datasets import load_dataset
from tqdm import tqdm
import gc
import warnings

warnings.filterwarnings("ignore")

# ==========================================
# 0. ENGINE SETUP & HYPERPARAMETERS
# ==========================================
device = "cuda" 
MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.2"

# HALO Coordinates
TARGET_LAYER = 16          # Mid-Layer (Knowledge Retrieval)
LATE_LAYER = -1            # Final Layer (Formatting/Output)

# Hardware/Compute Bounds
BEAM_WIDTH = 16            # Optimized from 64 for 75% Compute Savings
TARGET_OUTPUT_SIZE = 100   # Hard cutoff to prevent infinite loops

# Cascade Physics Limits
FRICTION_WEIGHT = 3.0      
VANGUARD_THRESHOLD = 1.5   # Triggers early O(N) Geometry Wakeup
STABILIZATION_EPSILON = 0.05 
GRAVITY_CONSTANT = 0.05    # Lexical Gravity (Soft Clamping)

# Syntactic Glue (Tokens that bypass heavy math to save compute)
STOP_WORDS = {"the", "a", "an", "is", "are", "was", "were", "and", "or", "but", "in", "on", "at", "to", "for", "of", "with", "by", "it", "this", "that", "these", "those", "from", "as", "be", "which", "who", "whom"}

print(f"Booting HALO V30 Ephemeral Engine on {MODEL_ID}...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, device_map="auto", torch_dtype=torch.bfloat16)
model.eval()

semantic_judge = SentenceTransformer('all-MiniLM-L6-v2', device=device)

# ==========================================
# 1. DYNAMIC DOMAIN CALIBRATION
# ==========================================
DOMAIN_ANCHORS = {
    "STEM": "The James Webb Space Telescope is a space telescope designed primarily to conduct infrared astronomy. As the largest telescope in space, its high resolution allows it to view objects too old or distant for the Hubble.",
    "Medical": "The human immune system is a complex network of cells and proteins that defends the body against infection. Leukocytes, or white blood cells, seek out and destroy disease-causing organisms or substances.",
    "Legal": "In criminal law, the prosecution must prove the defendant's guilt beyond a reasonable doubt. This high standard of proof ensures that the legal system protects the presumption of innocence until guilt is established.",
    "History": "The Industrial Revolution was the transition to new manufacturing processes in Great Britain, continental Europe, and the United States. It included going from hand production methods to machines and new iron production.",
    "Pop Culture": "In the classic fairy tale, Cinderella is a young woman living in unfortunate circumstances that are suddenly changed to remarkable fortune. The story involves a magical glass slipper and a royal ball.",
    "Epistemology": "The scientific method demands empirical evidence, falsifiability, and rigorous peer review to validate any claim. Pseudoscience, astrology, and superstitions rely on anecdotal evidence, cognitive biases, and unfalsifiable hypotheses, strictly failing to meet the rigid structural criteria of objective reality."
}

DOMAIN_THRESHOLDS = {
    "STEM": 1.9, "Medical": 1.8, "Legal": 1.8, "History": 1.9, 
    "Epistemology": 1.9, "Pop Culture": 2.5  
}

print("Pre-computing Domain Anchor Tensors...")
PRECOMPUTED_ANCHOR_EMBEDDINGS = {
    domain: semantic_judge.encode(text, convert_to_tensor=True) 
    for domain, text in DOMAIN_ANCHORS.items()
}

def extract_domain_physics(calibration_text, model, tokenizer, target_layer, late_layer):
    inputs = tokenizer(calibration_text, return_tensors="pt").to(device)
    with torch.no_grad():
        out = model(**inputs, output_hidden_states=True)
        
    hidden_states_mid = out.hidden_states[target_layer][0] 
    hidden_states_late = out.hidden_states[late_layer][0]
    
    variances, fluxes, curvatures, depths = [], [], [], []
    running_norm = 0.0
    recent_nodes = []
    prompt_anchor, last_node_v = None, None

    for i in range(hidden_states_mid.size(0)):
        # 100% GPU-Native Execution (Zero CPU Syncs)
        v = hidden_states_mid[i].detach().float().flatten()
        norm = torch.linalg.norm(v).item()
        u_v = v / (norm + 1e-9)
        
        v_late = hidden_states_late[i].detach().float().flatten()
        u_late = v_late / (torch.linalg.norm(v_late).item() + 1e-9)
        
        depths.append(1.0 - F.cosine_similarity(u_v, u_late, dim=0).item())
        
        if last_node_v is not None and prompt_anchor is not None:
            variances.append(abs(norm - running_norm) / (running_norm + 1e-9))
            fluxes.append(1.0 - F.cosine_similarity(u_v, last_node_v, dim=0).item())
            curvatures.append(1.0 - F.cosine_similarity(u_v, prompt_anchor, dim=0).item())
            
        recent_nodes.append(u_v)
        if len(recent_nodes) > 10: recent_nodes.pop(0)
            
        if prompt_anchor is None:
            prompt_anchor, last_node_v, running_norm = u_v, u_v, norm
        else:
            last_node_v = u_v
            running_norm = (0.9 * running_norm) + (0.1 * norm)
            prompt_anchor = F.normalize(torch.stack(recent_nodes).mean(dim=0), dim=0)

    return {
        'mu_var': np.mean(variances), 'sigma_var': max(0.02, np.std(variances)),
        'mu_flux': np.mean(fluxes), 'sigma_flux': max(0.02, np.std(fluxes)),
        'mu_curv': np.mean(curvatures), 'sigma_curv': max(0.02, np.std(curvatures)),
        'mu_depth': np.mean(depths), 'sigma_depth': max(0.02, np.std(depths))
    }

def route_and_calibrate_elastic(prompt, model, tokenizer, target_layer=16, late_layer=-1):
    prompt_emb = semantic_judge.encode(prompt, convert_to_tensor=True)
    scores = {d: util.cos_sim(prompt_emb, PRECOMPUTED_ANCHOR_EMBEDDINGS[d]).item() for d in DOMAIN_ANCHORS.keys()}
        
    sorted_domains = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:2]
    w1 = sorted_domains[0][1]
    w2 = sorted_domains[1][1] if len(sorted_domains) > 1 else 0
    total = w1 + w2 + 1e-9
    norm_w1, norm_w2 = w1 / total, w2 / total
    
    elastic_threshold = (DOMAIN_THRESHOLDS[sorted_domains[0][0]] * norm_w1) + \
                        (DOMAIN_THRESHOLDS[sorted_domains[1][0]] * norm_w2 if len(sorted_domains) > 1 else 0)
    
    prof_1 = extract_domain_physics(DOMAIN_ANCHORS[sorted_domains[0][0]], model, tokenizer, target_layer, late_layer)
    if len(sorted_domains) > 1 and norm_w2 > 0.1:
        prof_2 = extract_domain_physics(DOMAIN_ANCHORS[sorted_domains[1][0]], model, tokenizer, target_layer, late_layer)
        blended = {k: (prof_1[k] * norm_w1) + (prof_2[k] * norm_w2) for k in prof_1.keys()}
        blended['dynamic_threshold'] = elastic_threshold
        return blended
        
    prof_1['dynamic_threshold'] = elastic_threshold
    return prof_1

# ==========================================
# 2. EPHEMERAL CACHE GENERATOR (ROPE FIX)
# ==========================================
def build_ephemeral_cache(sacred_cache, bz):
    """
    Creates an isolated sandbox cache to completely prevent RoPE Positional Corruption.
    The Main KV Cache is never mutated in place.
    """
    if sacred_cache is None: return None
    ephemeral = DynamicCache()
    keys = getattr(sacred_cache, "key_cache", getattr(sacred_cache, "_key_cache", []))
    values = getattr(sacred_cache, "value_cache", getattr(sacred_cache, "_value_cache", []))
    
    for i in range(len(keys)):
        ephemeral.key_cache.append(keys[i].repeat(bz, 1, 1, 1).clone())
        ephemeral.value_cache.append(values[i].repeat(bz, 1, 1, 1).clone())
        
    if hasattr(sacred_cache, '_seen_tokens'):
        ephemeral._seen_tokens = sacred_cache._seen_tokens
        
    return ephemeral

# ==========================================
# 3. GPU-NATIVE 7-VERTEX TWIN (HALO CORE)
# ==========================================
class ErrorFieldTwinV30:
    def __init__(self, calibration_data, window_size=10, hidden_dim=4096):
        self.calib = calibration_data
        
        self.horizon_node = None       
        self.pivot_node = None         
        self.last_node_v = None        
        self.prompt_anchor = None      
        
        self.running_norm = 0.0
        self.window_size = window_size
        
        # Pre-allocated CUDA buffer for 100% GPU-native vertex tracking
        self.node_buffer = torch.zeros((window_size, hidden_dim), dtype=torch.float32, device=device)
        self.node_count = 0
        
        self.previous_flux = 0.0
        self.stable_ticks = 0
        self.guardian_active = False

    def calculate_friction(self, cand_hidden_state_mid, cand_hidden_state_late):
        if self.last_node_v is None or self.prompt_anchor is None: return 0.0 
            
        v = cand_hidden_state_mid.detach().float().flatten()
        norm = torch.linalg.norm(v).item()
        u_v = v / (norm + 1e-9)
        
        v_late = cand_hidden_state_late.detach().float().flatten()
        u_late = v_late / (torch.linalg.norm(v_late).item() + 1e-9)

        # RADAR A: DEPTH (Cross-Layer Schism)
        raw_depth = 1.0 - F.cosine_similarity(u_v, u_late, dim=0).item()
        z_depth = max(0.0, (raw_depth - self.calib['mu_depth']) / self.calib['sigma_depth'])

        # RADAR B: TIME (Kinematic Geometry)
        raw_variance = abs(norm - self.running_norm) / (self.running_norm + 1e-9)
        raw_flux = 1.0 - F.cosine_similarity(u_v, self.last_node_v, dim=0).item()
        raw_curvature = 1.0 - F.cosine_similarity(u_v, self.prompt_anchor, dim=0).item()
        
        z_var = max(0.0, (raw_variance - self.calib['mu_var']) / self.calib['sigma_var'])
        z_flux = max(0.0, (raw_flux - self.calib['mu_flux']) / self.calib['sigma_flux'])
        z_curv = max(0.0, (raw_curvature - self.calib['mu_curv']) / self.calib['sigma_curv'])
        base_z = max(z_var, z_flux, z_curv)

        delta_flux = abs(raw_flux - self.previous_flux)
        if not self.guardian_active:
            if delta_flux < STABILIZATION_EPSILON:
                self.stable_ticks += 1
                if self.stable_ticks >= 2: self.guardian_active = True
            else:
                self.stable_ticks = 0
            if not self.guardian_active: return z_depth 

        final_topological_z = base_z
        if (base_z >= VANGUARD_THRESHOLD or z_depth >= VANGUARD_THRESHOLD) and self.pivot_node is not None and self.horizon_node is not None:
            acceleration_penalty = delta_flux * 2.0 
            
            # Pure Tangent Extraction (No Broadcast Pollution)
            t1 = F.normalize(self.last_node_v - self.pivot_node, dim=0)
            t2 = F.normalize(u_v - self.last_node_v, dim=0)
            torsion = 1.0 - F.cosine_similarity(t2, t1, dim=0).item()
            drift = 1.0 - F.cosine_similarity(u_v, self.horizon_node, dim=0).item()
            
            final_topological_z = base_z + acceleration_penalty + (torsion * 1.5) + (drift * 0.5)

        # RADAR C: VOLUME (Semantic Collapse)
        z_vol = 0.0
        if self.node_count >= 3 and self.horizon_node is not None and self.pivot_node is not None:
            # Reconstruct the contiguous spatial gap using Circular Buffer
            v2 = self.node_buffer[(self.node_count - 3) % self.window_size]    
            
            vertices = torch.stack([self.horizon_node, self.prompt_anchor, v2, self.pivot_node, self.last_node_v, u_v])
            gram_matrix = torch.matmul(vertices, vertices.T)
            
            # CUDA Native Stabilization
            gram_matrix = gram_matrix + (torch.eye(6, device=device) * 1e-5)
            semantic_volume = torch.abs(torch.linalg.det(gram_matrix)).item()
            
            # Logarithmic Volume Decay Penalty
            if semantic_volume < 1e-10:
                z_vol = 3.0 
            else:
                log_vol = torch.log10(torch.tensor(semantic_volume, device=device))
                z_vol = max(0.0, (-3.0 - log_vol.item()) * 0.5)

        # THE HALO GATE
        return max(final_topological_z, z_depth, z_vol)

    def update_trajectory(self, hidden_state):
        v = hidden_state.detach().float().flatten()
        norm = torch.linalg.norm(v).item()
        u_v = v / (norm + 1e-9)
        
        if self.horizon_node is None:
            self.horizon_node = u_v
            
        if self.last_node_v is not None:
            self.previous_flux = 1.0 - F.cosine_similarity(u_v, self.last_node_v, dim=0).item()
            self.pivot_node = self.last_node_v 
        
        # Efficient GPU Circular Buffer Update
        idx = self.node_count % self.window_size
        self.node_buffer[idx] = u_v
        self.node_count += 1
        
        if self.prompt_anchor is None:
            self.prompt_anchor, self.last_node_v, self.running_norm = u_v, u_v, norm
        else:
            self.last_node_v = u_v
            self.running_norm = (0.9 * self.running_norm) + (0.1 * norm)
            
            # Rapid vector aggregation via GPU mean
            valid_nodes = min(self.node_count, self.window_size)
            self.prompt_anchor = F.normalize(torch.mean(self.node_buffer[:valid_nodes], dim=0), dim=0)

# ==========================================
# 4. SURGICAL PIPELINE (EPHEMERAL EXECUTION)
# ==========================================
def generate_v30_answer(prompt, calibration_data, use_hybrid=True):
    full_prompt = f"[INST] {prompt} [/INST]"
    inputs = tokenizer(full_prompt, return_tensors="pt").to(device)
    
    twin = ErrorFieldTwinV30(calibration_data=calibration_data)
    generated_ids = []
    dynamic_exec_limit = calibration_data.get('dynamic_threshold', 1.9)
    previous_str_raw = ""
    
    with torch.no_grad():
        out = model(**inputs, use_cache=True, output_hidden_states=True)
        sacred_cache = out.past_key_values
        next_logits = out.logits[:, -1, :]
        attention_mask = inputs.attention_mask
        if use_hybrid: twin.update_trajectory(out.hidden_states[TARGET_LAYER][0, -1, :])

    for _ in range(TARGET_OUTPUT_SIZE):
        topk = torch.topk(next_logits, BEAM_WIDTH, dim=-1)
        top1_idx = topk.indices[0][0]
        
        top1_str_raw = tokenizer.decode([top1_idx.item()])
        clean_str = top1_str_raw.strip().lower()
        is_glue = (len(clean_str) <= 1) or (clean_str in STOP_WORDS) or (top1_idx.item() == tokenizer.eos_token_id)
        
        combined_text = previous_str_raw + top1_str_raw
        if "\n\n" in combined_text or top1_str_raw == "\n":
            twin.horizon_node = None 
        previous_str_raw = top1_str_raw
        
        native_friction = 0.0
        if use_hybrid and not is_glue:
            # Native Candidate Test (Sandbox 1)
            with torch.no_grad():
                native_out = model(
                    input_ids=top1_idx.view(1, 1), 
                    attention_mask=torch.cat([attention_mask, torch.ones((1, 1), device=device)], dim=1), 
                    past_key_values=build_ephemeral_cache(sacred_cache, 1), 
                    use_cache=False, output_hidden_states=True
                )
            native_friction = twin.calculate_friction(
                native_out.hidden_states[TARGET_LAYER][0, 0, :],
                native_out.hidden_states[LATE_LAYER][0, 0, :]
            )
            del native_out
        
        if not use_hybrid or is_glue or native_friction <= dynamic_exec_limit:
            chosen_id = top1_idx.view(1, 1)
        else:
            # STAGE 3: THE EPHEMERAL SANDBOX (Beam Context Surgery)
            batch_ids = topk.indices[0].view(BEAM_WIDTH, 1)
            ephemeral_mask = torch.cat([attention_mask.repeat(BEAM_WIDTH, 1), torch.ones((BEAM_WIDTH, 1), device=device)], dim=1)
            ephemeral_cache = build_ephemeral_cache(sacred_cache, BEAM_WIDTH)
            
            with torch.no_grad():
                sandbox_out = model(
                    input_ids=batch_ids, 
                    attention_mask=ephemeral_mask, 
                    past_key_values=ephemeral_cache, 
                    use_cache=False, 
                    output_hidden_states=True
                )
                
            raw_logits = next_logits[0, topk.indices[0]].clone().float()
            frictions = torch.zeros(BEAM_WIDTH, device=device)
            
            for k in range(BEAM_WIDTH):
                frictions[k] = twin.calculate_friction(
                    sandbox_out.hidden_states[TARGET_LAYER][k, 0, :],
                    sandbox_out.hidden_states[LATE_LAYER][k, 0, :]
                )
            
            # Soft Clamping (Lexical Gravity)
            rank_penalty = torch.arange(BEAM_WIDTH, device=device).float() * GRAVITY_CONSTANT
            clamped_frictions = torch.clamp(frictions - dynamic_exec_limit, min=0.0)
            adjusted_logits = raw_logits - (clamped_frictions * FRICTION_WEIGHT) - rank_penalty
            
            winner_idx = torch.argmax(adjusted_logits).item()
            chosen_id = topk.indices[0][winner_idx].view(1, 1)
            
            # The Sandbox is immediately deleted here.
            del sandbox_out, ephemeral_cache
            torch.cuda.empty_cache()

        # ADVANCE THE SACRED TIMELINE (Batch=1)
        # Hugging Face safely auto-handles the RoPE embeddings natively.
        with torch.no_grad():
            attention_mask = torch.cat([attention_mask, torch.ones((1, 1), device=device)], dim=1)
            out = model(
                input_ids=chosen_id, 
                attention_mask=attention_mask, 
                past_key_values=sacred_cache, 
                use_cache=True, 
                output_hidden_states=True
            )
            sacred_cache = out.past_key_values
            next_logits = out.logits[:, -1, :]
            if use_hybrid: twin.update_trajectory(out.hidden_states[TARGET_LAYER][0, -1, :])

        generated_ids.append(chosen_id.item())
        if chosen_id.item() == tokenizer.eos_token_id: break

    return tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

# ==========================================
# 5. EXECUTION & BENCHMARKING
# ==========================================
if __name__ == "__main__":
    dataset = load_dataset("truthful_qa", "generation", split="validation")
    results = []
    
    for i in tqdm(range(len(dataset)), desc="HALO V30 Execution"):
        item = dataset[i]
        q = item['question']
        
        dynamic_calib = route_and_calibrate_elastic(q, model, tokenizer, TARGET_LAYER, LATE_LAYER)
        base = generate_v30_answer(q, dynamic_calib, use_hybrid=False)
        steer = generate_v30_answer(q, dynamic_calib, use_hybrid=True)
        
        results.append({
            'question': q, 
            'baseline': base, 
            'steered': steer
        })
        
        torch.cuda.empty_cache()
        gc.collect()

        # Save Checkpoints
        if (i + 1) % 5 == 0:
            pd.DataFrame(results).to_csv("halo_engine_v30_output.csv", index=False)

    pd.DataFrame(results).to_csv("halo_engine_v30_output.csv", index=False)
    print(f"\n[SYSTEM SECURED] HALO Architecture V30 Processing Complete.")

In [ ]:
import os
import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer, LogitsProcessor, LogitsProcessorList
from datasets import load_dataset
from tqdm import tqdm
import gc
import warnings

warnings.filterwarnings("ignore")

# ==========================================
# 0. ENGINE SETUP (Hardware Agnostic)
# ==========================================
MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.2"

TARGET_LAYER = 16          
LATE_LAYER = -1            
TARGET_OUTPUT_SIZE = 100   

VANGUARD_THRESHOLD_VAL = 1.5   
STABILIZATION_EPSILON_VAL = 0.05 

print(f"Booting HALO V40.1 (OOM-Proof Epistemic Engine) on {MODEL_ID}...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, device_map="auto", torch_dtype=torch.bfloat16)
model.eval()

INPUT_DEVICE = next(model.parameters()).device

# ==========================================
# 0.5. CHUNKED LM HEAD EXTRACTION (OOM-SAFE)
# ==========================================
print("Extracting LM Head weights via Chunked Identity Matrix...")
lm_head_module = model.get_output_embeddings()
hidden_dim = lm_head_module.in_features
vocab_size = lm_head_module.out_features

# Pre-allocate CPU tensor for the final normalized weights
raw_lm_head = torch.empty((vocab_size, hidden_dim), device='cpu', dtype=torch.float32)
chunk_size = 256 # Reduces VRAM spike from 250MB to ~16MB

with torch.no_grad():
    # Capture bias safely (Mistral has no bias, but this ensures universal compatibility)
    zeros = torch.zeros((1, hidden_dim), device=INPUT_DEVICE, dtype=model.dtype)
    b = lm_head_module(zeros)
    
    for i in tqdm(range(0, hidden_dim, chunk_size), desc="Extracting Matrix"):
        end = min(i + chunk_size, hidden_dim)
        sz = end - i
        
        identity_chunk = torch.zeros((sz, hidden_dim), device=INPUT_DEVICE, dtype=model.dtype)
        identity_chunk[torch.arange(sz), torch.arange(i, end)] = 1.0
        
        # Forward pass just the chunk
        chunk_out = lm_head_module(identity_chunk)
        w_chunk = (chunk_out - b).t().float().cpu()
        
        raw_lm_head[:, i:end] = w_chunk
        del identity_chunk, chunk_out, w_chunk
        
    del zeros, b
    torch.cuda.empty_cache()

# Normalize once globally
GLOBAL_LM_HEAD_NORMALIZED = F.normalize(raw_lm_head, dim=-1)
del raw_lm_head
gc.collect()
print("Global LM Head extraction complete.")

# ==========================================
# 1. EXTERNAL TRUTH DATASTORE (Production FAISS)
# ==========================================
class FAISSDatastore:
    def __init__(self, vocab_size, index_path=None, token_map_path=None, k=16, temperature=0.5):
        self.vocab_size = vocab_size
        self.k = k
        self.temperature = temperature
        self.is_active = False
        
        if index_path and os.path.exists(index_path) and token_map_path and os.path.exists(token_map_path):
            import faiss 
            print(f"Loading FAISS Datastore from {index_path}...")
            self.index = faiss.read_index(index_path)
            self.token_map = torch.load(token_map_path) 
            self.is_active = True
        else:
            print("FAISS Index not found. Defaulting to Uniform Prior (Neutral Truth Signal).")

    def lookup(self, hidden_state):
        if not self.is_active:
            return torch.ones(self.vocab_size, device=hidden_state.device) / self.vocab_size
            
        # FIX: Cast bfloat16 to float32 inside PyTorch BEFORE calling .numpy()
        hidden_np = hidden_state.detach().cpu().float().numpy()
        
        distances, indices = self.index.search(hidden_np, self.k)
        weights = F.softmax(-torch.tensor(distances[0]) / self.temperature, dim=0)
        retrieved_tokens = self.token_map[indices[0]]
        
        p_truth = torch.zeros(self.vocab_size, device=hidden_state.device)
        p_truth.scatter_add_(0, retrieved_tokens.to(hidden_state.device), weights.to(hidden_state.device))
        
        return p_truth

# ==========================================
# 2. DYNAMIC DOMAIN CALIBRATION
# ==========================================
def route_and_calibrate_elastic(prompt):
    return {
        'mu_var': 0.1, 'sigma_var': 0.05, 'mu_flux': 0.2, 'sigma_flux': 0.05,
        'mu_curv': 0.3, 'sigma_curv': 0.08, 'mu_depth': 0.4, 'sigma_depth': 0.1,
        'dynamic_threshold': 1.9 
    }

# ==========================================
# 3. THE ROBUST INTERCEPTOR
# ==========================================
class HiddenStateInterceptor:
    def __init__(self, model, target_layer_idx):
        self.states = {}
        self.hooks = []
        
        layers_module = getattr(model.model, "layers", None)
        norm_module = getattr(model.model, "norm", getattr(model.model, "ln_f", None))
                
        def get_mid_hook():
            def hook(module, args, output):
                state = output[0] if isinstance(output, tuple) else output
                self.states['mid'] = state[:, -1, :].detach()
            return hook
            
        def get_late_hook():
            def hook(module, args, output):
                state = output[0] if isinstance(output, tuple) else output
                self.states['late'] = state[:, -1, :].detach()
            return hook

        if layers_module is not None:
            self.hooks.append(layers_module[target_layer_idx].register_forward_hook(get_mid_hook()))
        if norm_module is not None:
            self.hooks.append(norm_module.register_forward_hook(get_late_hook()))
        
    def remove(self):
        for h in self.hooks: h.remove()

# ==========================================
# 4. FULLY TENSORIZED 7-VERTEX TWIN
# ==========================================
class ErrorFieldTwinV40:
    def __init__(self, calibration_data, window_size=10, hidden_dim=4096):
        self.calib_raw = calibration_data
        self.window_size = window_size
        self.hidden_dim = hidden_dim
        self.device_initialized = False
        
        self.horizon_node = None       
        self.pivot_node = None         
        self.last_node_v = None        
        self.prompt_anchor = None      
        
        self.stable_ticks = 0
        self.guardian_active = False

    def _initialize_device_tensors(self, local_device):
        self.local_device = local_device
        self.calib = {k: torch.tensor(v, device=local_device) for k, v in self.calib_raw.items()}
        
        self.running_norm = torch.tensor(0.0, device=local_device)
        self.node_buffer = torch.zeros((self.window_size, self.hidden_dim), dtype=torch.float32, device=local_device)
        self.node_count = 0
        self.previous_flux = torch.tensor(0.0, device=local_device)
        
        self.vanguard_thresh = torch.tensor(VANGUARD_THRESHOLD_VAL, device=local_device)
        self.stab_epsilon = torch.tensor(STABILIZATION_EPSILON_VAL, device=local_device)
        self.device_initialized = True

    def set_barycentric_horizon(self, prompt_hidden_states):
        local_device = prompt_hidden_states.device
        if not self.device_initialized:
            self._initialize_device_tensors(local_device)
            
        centroid = prompt_hidden_states.mean(dim=1)[0].float()
        self.horizon_node = F.normalize(centroid, dim=0)

    def calculate_friction_and_update(self, cand_mid, cand_late):
        local_device = cand_mid.device
        if not self.device_initialized:
            self._initialize_device_tensors(local_device)
            
        v = cand_mid.float()
        norm = torch.linalg.norm(v)
        u_v = v / (norm + 1e-9)
        
        v_late = cand_late.to(local_device).float()
        u_late = v_late / (torch.linalg.norm(v_late) + 1e-9)

        if self.last_node_v is None or self.prompt_anchor is None:
            self._update_trajectory(u_v, norm)
            return torch.tensor(0.0, device=local_device)
            
        # DYNAMIC DEVICE SYNCHRONIZATION: Immunize against HF device_map shifting
        mu_depth = self.calib['mu_depth'].to(local_device)
        sigma_depth = self.calib['sigma_depth'].to(local_device)
        mu_var = self.calib['mu_var'].to(local_device)
        sigma_var = self.calib['sigma_var'].to(local_device)
        mu_flux = self.calib['mu_flux'].to(local_device)
        sigma_flux = self.calib['sigma_flux'].to(local_device)
        mu_curv = self.calib['mu_curv'].to(local_device)
        sigma_curv = self.calib['sigma_curv'].to(local_device)
        
        running_norm_local = self.running_norm.to(local_device)
        prev_flux_local = self.previous_flux.to(local_device)
        vang_thresh_local = self.vanguard_thresh.to(local_device)
        stab_eps_local = self.stab_epsilon.to(local_device)
        
        last_node_local = self.last_node_v.to(local_device)
        anchor_local = self.prompt_anchor.to(local_device)

        raw_depth = 1.0 - F.cosine_similarity(u_v, u_late, dim=0)
        z_depth = torch.clamp((raw_depth - mu_depth) / sigma_depth, min=0.0)

        raw_variance = torch.abs(norm - running_norm_local) / (running_norm_local + 1e-9)
        raw_flux = 1.0 - F.cosine_similarity(u_v, last_node_local, dim=0)
        raw_curv = 1.0 - F.cosine_similarity(u_v, anchor_local, dim=0)
        
        z_var = torch.clamp((raw_variance - mu_var) / sigma_var, min=0.0)
        z_flux = torch.clamp((raw_flux - mu_flux) / sigma_flux, min=0.0)
        z_curv = torch.clamp((raw_curv - mu_curv) / sigma_curv, min=0.0)
        
        base_z = torch.max(torch.stack([z_var, z_flux, z_curv]))
        delta_flux = torch.abs(raw_flux - prev_flux_local)
        
        if not self.guardian_active:
            if delta_flux < stab_eps_local:
                self.stable_ticks += 1
                if self.stable_ticks >= 2: self.guardian_active = True
            else:
                self.stable_ticks = 0
            if not self.guardian_active:
                self._update_trajectory(u_v, norm)
                return z_depth

        final_topo_z = base_z
        if (base_z >= vang_thresh_local or z_depth >= vang_thresh_local) and self.pivot_node is not None and self.horizon_node is not None:
            accel_penalty = delta_flux * 2.0
            t1 = F.normalize(last_node_local - self.pivot_node.to(local_device), dim=0)
            t2 = F.normalize(u_v - last_node_local, dim=0)
            torsion = 1.0 - F.cosine_similarity(t2, t1, dim=0)
            drift = 1.0 - F.cosine_similarity(u_v, self.horizon_node.to(local_device), dim=0)
            final_topo_z = base_z + accel_penalty + (torsion * 1.5) + (drift * 0.5)

        z_vol = torch.tensor(0.0, device=local_device)
        if self.node_count >= 3 and self.horizon_node is not None and self.pivot_node is not None:
            v2 = self.node_buffer[(self.node_count - 3) % self.window_size].to(local_device)
            vertices = torch.stack([
                self.horizon_node.to(local_device), 
                anchor_local, 
                v2, 
                self.pivot_node.to(local_device), 
                last_node_local, 
                u_v
            ])
            gram = torch.matmul(vertices, vertices.T)
            gram = gram + (torch.eye(6, device=local_device) * 1e-5)
            
            vol = torch.clamp(torch.abs(torch.linalg.det(gram)), min=1e-12)
            if vol < 1e-10:
                z_vol = torch.tensor(3.0, device=local_device)
            else:
                log_vol = torch.log10(vol)
                z_vol = torch.clamp((-3.0 - log_vol) * 0.5, min=0.0)

        self._update_trajectory(u_v, norm)
        return torch.max(torch.stack([final_topo_z, z_depth, z_vol]))

    def _update_trajectory(self, u_v, norm):
        local_device = u_v.device
        
        if self.horizon_node is not None:
            self.horizon_node = self.horizon_node.to(local_device)
            alpha = 0.9 if self.node_count > 5 else 0.7
            self.horizon_node = F.normalize((alpha * self.horizon_node) + ((1.0 - alpha) * u_v), dim=0)
            
            if self.node_count > 0 and self.node_count % 20 == 0 and self.prompt_anchor is not None:
                self.prompt_anchor = self.prompt_anchor.to(local_device)
                self.horizon_node = F.normalize((0.7 * self.horizon_node) + (0.3 * self.prompt_anchor), dim=0)
        else:
            self.horizon_node = u_v

        if self.last_node_v is not None:
            self.last_node_v = self.last_node_v.to(local_device)
            self.previous_flux = 1.0 - F.cosine_similarity(u_v, self.last_node_v, dim=0)
            self.pivot_node = self.last_node_v
            
        idx = self.node_count % self.window_size
        self.node_buffer[idx] = u_v
        self.node_count += 1
        
        if self.prompt_anchor is None:
            self.prompt_anchor, self.last_node_v, self.running_norm = u_v, u_v, norm
        else:
            self.last_node_v = u_v
            self.running_norm = (0.9 * self.running_norm.to(local_device)) + (0.1 * norm.to(local_device))
            valid_nodes = min(self.node_count, self.window_size)
            self.prompt_anchor = F.normalize(torch.mean(self.node_buffer[:valid_nodes], dim=0), dim=0)

# ==========================================
# 5. SOTA EPISTEMIC CONTROLLER (V40.1)
# ==========================================
class HaloEpistemicProcessor(LogitsProcessor):
    def __init__(self, twin, interceptor, exec_limit_val, raw_lm_head_normalized, truth_datastore=None):
        self.twin = twin
        self.interceptor = interceptor
        self.raw_lm_head = raw_lm_head_normalized
        self.truth_datastore = truth_datastore
        self.lm_head_cache = {} 
        self.exec_limit_tensor = None
        self.max_entropy = None

    def __call__(self, input_ids, scores):
        with torch.no_grad():
            mid_state = self.interceptor.states.get('mid')
            late_state = self.interceptor.states.get('late')
            
            target_device = scores.device
            
            if self.exec_limit_tensor is None:
                self.exec_limit_tensor = torch.tensor(1.9, device=target_device)
                self.approx_k = min(512, scores.shape[-1])
                self.max_entropy = torch.log(torch.tensor(self.approx_k, dtype=torch.float32, device=target_device))
                
            if target_device not in self.lm_head_cache:
                self.lm_head_cache[target_device] = self.raw_lm_head.to(target_device)
            local_lm_head = self.lm_head_cache[target_device]
            
            if mid_state is not None and late_state is not None:
                base_friction = self.twin.calculate_friction_and_update(mid_state[0], late_state[0]).to(target_device)
                
                if base_friction > self.exec_limit_tensor and self.twin.horizon_node is not None:
                    
                    k = min(128, scores.shape[-1])
                    top_indices = torch.topk(scores[0], k).indices
                    candidate_vecs = local_lm_head[top_indices]
                    
                    horizon = self.twin.horizon_node.to(target_device)
                    token_drift = 1.0 - F.cosine_similarity(candidate_vecs, horizon.unsqueeze(0), dim=-1)
                    
                    drift_mean = token_drift.mean()
                    drift_std = torch.clamp(token_drift.std(), min=1e-4)
                    adaptive_threshold = drift_mean + (1.0 * drift_std)
                    
                    top_scores_ent, _ = torch.topk(scores[0], self.approx_k)
                    probs = F.softmax(top_scores_ent - top_scores_ent.max(), dim=-1)
                    entropy = -torch.sum(probs * torch.log(probs + 1e-9), dim=-1)
                    
                    entropy_ratio = torch.clamp(entropy / self.max_entropy, max=1.0)
                    gate = torch.clamp(1.0 - entropy_ratio, min=0.1)
                    
                    dynamic_gate_threshold = 0.2 + 0.3 * (base_friction / self.exec_limit_tensor)
                    extreme_scale = 1.5 + 0.5 * (base_friction / self.exec_limit_tensor)
                    extreme_mask = token_drift > (adaptive_threshold * extreme_scale)
                    
                    effective_mask = (token_drift > adaptive_threshold) & ((gate > dynamic_gate_threshold) | extreme_mask)
                    
                    if effective_mask.sum() > (0.8 * k):
                        effective_mask[:] = False 
                    
                    if effective_mask.any():
                        token_weights = (token_drift / (token_drift.max() + 1e-6)) ** 1.5
                        
                        if self.truth_datastore and self.truth_datastore.is_active:
                            p_truth = self.truth_datastore.lookup(mid_state[0]).to(target_device)
                            raw_truth_support = p_truth[top_indices]
                            
                            truth_support = torch.clamp(raw_truth_support, min=1e-6)
                            truth_support = truth_support / (truth_support.sum() + 1e-9)
                            
                            max_truth = truth_support.max() + 1e-9
                            truth_discount = truth_support / max_truth
                        else:
                            truth_discount = torch.zeros(k, device=target_device)

                        pruning_indices = top_indices[effective_mask]
                        final_penalties = 20.0 * gate * token_weights[effective_mask] * (1.0 - truth_discount[effective_mask])
                        scores[0, pruning_indices] -= final_penalties
                    
        return scores

# NOTE: We now pass GLOBAL_LM_HEAD_NORMALIZED directly
def generate_v40_answer(prompt, calibration_data, lm_head_normalized, truth_datastore=None, use_hybrid=True):
    full_prompt = f"[INST] {prompt} [/INST]"
    
    inputs = tokenizer(full_prompt, return_tensors="pt")
    inputs = {k: v.to(INPUT_DEVICE) for k, v in inputs.items()}
    
    twin = ErrorFieldTwinV40(calibration_data=calibration_data)
    interceptor = HiddenStateInterceptor(model, TARGET_LAYER)
    dynamic_exec_limit_val = calibration_data.get('dynamic_threshold', 1.9)
    
    processors = LogitsProcessorList()
    if use_hybrid:
        processors.append(HaloEpistemicProcessor(twin, interceptor, dynamic_exec_limit_val, lm_head_normalized, truth_datastore))
        
    try:
        with torch.no_grad():
            pre_flight = model(**inputs, output_hidden_states=True)
            prompt_hidden_states = pre_flight.hidden_states[TARGET_LAYER].detach()
            twin.set_barycentric_horizon(prompt_hidden_states)
            del pre_flight
            
            output_ids = model.generate(
                **inputs,
                max_new_tokens=TARGET_OUTPUT_SIZE,
                logits_processor=processors,
                use_cache=True, 
                pad_token_id=tokenizer.eos_token_id,
                do_sample=False 
            )
    finally:
        interceptor.remove() 
        
    generated_ids = output_ids[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

# ==========================================
# 6. EXECUTION & BENCHMARKING
# ==========================================
if __name__ == "__main__":
    dataset = load_dataset("truthful_qa", "generation", split="validation")
    results = []
    
    #truth_db = FAISSDatastore(vocab_size=len(tokenizer)) 
    truth_db = FAISSDatastore(
        vocab_size=len(tokenizer), 
        index_path="tier2_fever.index",          # <--- Just the filename
        token_map_path="tier2_fever_token_map.pt" # <--- Just the filename
    )
    
    for i in tqdm(range(len(dataset)), desc="HALO V40.1 Apex Benchmark"):
        item = dataset[i]
        q = item['question']
        
        dynamic_calib = route_and_calibrate_elastic(q)
        
        # We pass the pre-computed GLOBAL_LM_HEAD_NORMALIZED to prevent memory leaks
        base = generate_v40_answer(q, dynamic_calib, GLOBAL_LM_HEAD_NORMALIZED, truth_datastore=truth_db, use_hybrid=False)
        steer = generate_v40_answer(q, dynamic_calib, GLOBAL_LM_HEAD_NORMALIZED, truth_datastore=truth_db, use_hybrid=True)
        
        results.append({'question': q, 'baseline': base, 'steered': steer})
        
        torch.cuda.empty_cache()
        gc.collect()

        if (i + 1) % 5 == 0:
            pd.DataFrame(results).to_csv("halo_engine_v40_1_output.csv", index=False)

    pd.DataFrame(results).to_csv("halo_engine_v40_1_output.csv", index=False)
    print(f"\n[SYSTEM SECURED] HALO Architecture V40.1 Processing Complete.")

In [ ]:
!pip install "datasets<3.0.0" faiss-cpu

In [ ]:
import os
import torch
import torch.nn.functional as F
import numpy as np
import faiss
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
from tqdm import tqdm
import gc

# ==========================================
# CONFIGURATION
# ==========================================
MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.2"
TARGET_LAYER = 16
CHUNK_SIZE = 256  # Tokens per forward pass
MAX_TOKENS_PER_INDEX = 500000  # Adjust based on your RAM (500k = ~4GB index)
DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"

print(f"Initializing V41 Datastore Compiler on {DEVICE}...")

# 1. Load Tokenizer & Model
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, device_map="auto", torch_dtype=torch.bfloat16)
model.eval()

# ==========================================
# PHASE 1: CORPUS ASSEMBLY
# ==========================================
def load_epistemic_corpus():
    """
    Pulls verified text from external datasets.
    For this build, we use FEVER (Fact Verification) and Wikipedia.
    """
    print("Downloading FEVER dataset (Evidence Layer)...")
    # We use the paperver subset of FEVER which contains claims and evidence
    fever = load_dataset("fever", "v1.0", split="train[:5000]", trust_remote_code=True)
    
    corpus_texts = []
    
    # Process FEVER: We only want SUPPORTED claims to act as absolute truth anchors
    for item in fever:
        if item['label'] == 'SUPPORTS':
            # Format: "Fact: [Claim]"
            corpus_texts.append(f"Fact: {item['claim']}")
            
    # Note: In a full production run, you would append Wikipedia and Wikidata subsets here.
    text_blob = " ".join(corpus_texts)
    
    # Tokenize the entire blob
    print("Tokenizing Corpus...")
    tokens = tokenizer(text_blob, return_tensors="pt").input_ids[0]
    print(f"Total Verified Tokens to Index: {len(tokens)}")
    
    return tokens

# ==========================================
# PHASE 2: MISTRAL VECTORIZATION
# ==========================================
def extract_hidden_states(tokens):
    """
    Passes tokens through Mistral in chunks.
    Extracts the Layer 16 hidden state for token T, and maps it to token T+1.
    """
    total_tokens = min(len(tokens) - 1, MAX_TOKENS_PER_INDEX)
    hidden_dim = model.config.hidden_size
    
    # Pre-allocate CPU tensors to prevent GPU OOM
    all_hidden_states = torch.empty((total_tokens, hidden_dim), dtype=torch.float32)
    all_next_tokens = torch.empty((total_tokens,), dtype=torch.long)
    
    current_idx = 0
    
    print(f"Extracting Layer {TARGET_LAYER} Geometry...")
    for i in tqdm(range(0, total_tokens, CHUNK_SIZE)):
        end_idx = min(i + CHUNK_SIZE, total_tokens)
        
        # We need chunk + 1 to get the "next token" for the last item in the chunk
        chunk_input = tokens[i : end_idx + 1].unsqueeze(0).to(DEVICE)
        
        with torch.no_grad():
            outputs = model(chunk_input, output_hidden_states=True)
            # Extract Layer 16. Shape: [1, seq_len, 4096]
            layer_16_states = outputs.hidden_states[TARGET_LAYER][0] 
            
            # Normalize the hidden states so we can use Inner Product (Cosine Similarity) in FAISS
            normalized_states = F.normalize(layer_16_states[:-1], dim=-1).cpu().float()
            
            # The target is the token immediately following the hidden state
            target_tokens = chunk_input[0, 1:].cpu()
            
            batch_size = normalized_states.shape[0]
            all_hidden_states[current_idx : current_idx + batch_size] = normalized_states
            all_next_tokens[current_idx : current_idx + batch_size] = target_tokens
            
            current_idx += batch_size
            
            del outputs, layer_16_states, normalized_states, target_tokens, chunk_input
            if i % (CHUNK_SIZE * 10) == 0:
                torch.cuda.empty_cache()
                
    return all_hidden_states[:current_idx], all_next_tokens[:current_idx]

# ==========================================
# PHASE 3: FAISS COMPILATION
# ==========================================
def build_and_save_index(hidden_states, next_tokens, index_name="tier2_fever"):
    print(f"\nBuilding FAISS Index: {index_name}...")
    
    hidden_dim = hidden_states.shape[1]
    hidden_np = hidden_states.numpy()
    
    # We use IndexFlatIP (Inner Product). Because we L2-normalized the vectors 
    # during extraction, Inner Product is mathematically identical to Cosine Similarity.
    index = faiss.IndexFlatIP(hidden_dim)
    
    print("Adding vectors to FAISS...")
    index.add(hidden_np)
    
    print("Saving to disk...")
    faiss.write_index(index, f"{index_name}.index")
    torch.save(next_tokens, f"{index_name}_token_map.pt")
    
    print(f"Successfully compiled {index_name}.index and mapped {len(next_tokens)} tokens.")

# ==========================================
# EXECUTION
# ==========================================
if __name__ == "__main__":
    # 1. Gather Data
    corpus_tokens = load_epistemic_corpus()
    
    # 2. Vectorize through Mistral
    h_states, target_tokens = extract_hidden_states(corpus_tokens)
    
    # 3. Compile FAISS Index
    build_and_save_index(h_states, target_tokens, index_name="tier2_fever")
    
    print("\n[PHASE 1 & 2 COMPLETE] Datastore is ready for V41 Integration.")

In [ ]:
import os
from IPython.display import FileLink, display

# Check the root directory for the files
print("Checking root directory...")
files_in_root = os.listdir('.')
print(files_in_root)

if "tier2_fever.index" in files_in_root and "tier2_fever_token_map.pt" in files_in_root:
    print("\nFiles found! Click the links below to download:")
    display(FileLink("tier2_fever.index"))
    display(FileLink("tier2_fever_token_map.pt"))
else:
    print("\nFiles not found. Did the script finish executing completely?")

In [ ]:
import os
import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer, LogitsProcessor, LogitsProcessorList
from datasets import load_dataset
from tqdm import tqdm
import gc
import warnings

warnings.filterwarnings("ignore")

# ==========================================
# 0. CONFIGURATION & FILE PATHS
# ==========================================
MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.2"

# ---> PASTE YOUR COMPLETE ABSOLUTE PATHS HERE <---
# Example for Windows: "C:/Users/Abhishek/Downloads/tier2_fever.index"
# Example for Colab/Kaggle: "/kaggle/input/my-dataset/tier2_fever.index"
INDEX_FILE_PATH = "/kaggle/input/datasets/omrai1/faiss-data/tier2_fever.index"
TOKEN_MAP_FILE_PATH = "/kaggle/input/datasets/omrai1/faiss-data/tier2_fever_token_map.pt"

TARGET_LAYER = 16          
LATE_LAYER = -1            
TARGET_OUTPUT_SIZE = 100   

VANGUARD_THRESHOLD_VAL = 1.5   
STABILIZATION_EPSILON_VAL = 0.05 

print(f"Booting HALO V41 (The Epistemic Engine) on {MODEL_ID}...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, device_map="auto", torch_dtype=torch.bfloat16)
model.eval()

INPUT_DEVICE = next(model.parameters()).device

# ==========================================
# 0.5. CHUNKED LM HEAD EXTRACTION
# ==========================================
print("Extracting LM Head weights via Chunked Identity Matrix...")
lm_head_module = model.get_output_embeddings()
hidden_dim = lm_head_module.in_features
vocab_size = lm_head_module.out_features

raw_lm_head = torch.empty((vocab_size, hidden_dim), device='cpu', dtype=torch.float32)
chunk_size = 256 

with torch.no_grad():
    zeros = torch.zeros((1, hidden_dim), device=INPUT_DEVICE, dtype=model.dtype)
    b = lm_head_module(zeros)
    
    for i in tqdm(range(0, hidden_dim, chunk_size), desc="Extracting Matrix"):
        end = min(i + chunk_size, hidden_dim)
        sz = end - i
        
        identity_chunk = torch.zeros((sz, hidden_dim), device=INPUT_DEVICE, dtype=model.dtype)
        identity_chunk[torch.arange(sz), torch.arange(i, end)] = 1.0
        
        chunk_out = lm_head_module(identity_chunk)
        w_chunk = (chunk_out - b).t().float().cpu()
        
        raw_lm_head[:, i:end] = w_chunk
        del identity_chunk, chunk_out, w_chunk
        
    del zeros, b
    torch.cuda.empty_cache()

GLOBAL_LM_HEAD_NORMALIZED = F.normalize(raw_lm_head, dim=-1)
del raw_lm_head
gc.collect()

# ==========================================
# 1. EXTERNAL TRUTH DATASTORE (V41 FEVER Integrator)
# ==========================================
class FAISSDatastore:
    def __init__(self, vocab_size, index_path, token_map_path, k=16, temperature=0.5):
        self.vocab_size = vocab_size
        self.k = k
        self.temperature = temperature
        self.is_active = False
        
        print(f"Verifying Datastore Paths...")
        print(f"Index: {index_path}")
        print(f"Map:   {token_map_path}")
        
        if os.path.exists(index_path) and os.path.exists(token_map_path):
            import faiss 
            print("Paths verified. Loading V41 FAISS Datastore into memory...")
            self.index = faiss.read_index(index_path)
            self.token_map = torch.load(token_map_path) 
            self.is_active = True
            print("Datastore Armed and Active.")
        else:
            print("[WARNING] Files not found at specified paths! Engine will run in pure Geometry Mode (V40.1).")

    def lookup(self, hidden_state):
        if not self.is_active:
            return torch.ones(self.vocab_size, device=hidden_state.device) / self.vocab_size
            
        # V41 FIX: Cast to float32 AND reshape to 2D matrix (1, dim) for FAISS
        hidden_np = hidden_state.detach().cpu().float().numpy().reshape(1, -1)
        
        distances, indices = self.index.search(hidden_np, self.k)
        weights = F.softmax(-torch.tensor(distances[0]) / self.temperature, dim=0)
        retrieved_tokens = self.token_map[indices[0]]
        
        p_truth = torch.zeros(self.vocab_size, device=hidden_state.device)
        p_truth.scatter_add_(0, retrieved_tokens.to(hidden_state.device), weights.to(hidden_state.device))
        
        return p_truth

# ==========================================
# 2. DYNAMIC DOMAIN CALIBRATION
# ==========================================
def route_and_calibrate_elastic(prompt):
    return {
        'mu_var': 0.1, 'sigma_var': 0.05, 'mu_flux': 0.2, 'sigma_flux': 0.05,
        'mu_curv': 0.3, 'sigma_curv': 0.08, 'mu_depth': 0.4, 'sigma_depth': 0.1,
        'dynamic_threshold': 1.9 
    }

# ==========================================
# 3. THE ROBUST INTERCEPTOR
# ==========================================
class HiddenStateInterceptor:
    def __init__(self, model, target_layer_idx):
        self.states = {}
        self.hooks = []
        
        layers_module = getattr(model.model, "layers", None)
        norm_module = getattr(model.model, "norm", getattr(model.model, "ln_f", None))
                
        def get_mid_hook():
            def hook(module, args, output):
                state = output[0] if isinstance(output, tuple) else output
                self.states['mid'] = state[:, -1, :].detach()
            return hook
            
        def get_late_hook():
            def hook(module, args, output):
                state = output[0] if isinstance(output, tuple) else output
                self.states['late'] = state[:, -1, :].detach()
            return hook

        if layers_module is not None:
            self.hooks.append(layers_module[target_layer_idx].register_forward_hook(get_mid_hook()))
        if norm_module is not None:
            self.hooks.append(norm_module.register_forward_hook(get_late_hook()))
        
    def remove(self):
        for h in self.hooks: h.remove()

# ==========================================
# 4. FULLY TENSORIZED 7-VERTEX TWIN
# ==========================================
class ErrorFieldTwinV40:
    def __init__(self, calibration_data, window_size=10, hidden_dim=4096):
        self.calib_raw = calibration_data
        self.window_size = window_size
        self.hidden_dim = hidden_dim
        self.device_initialized = False
        
        self.horizon_node = None       
        self.pivot_node = None         
        self.last_node_v = None        
        self.prompt_anchor = None      
        
        self.stable_ticks = 0
        self.guardian_active = False

    def _initialize_device_tensors(self, local_device):
        self.local_device = local_device
        self.calib = {k: torch.tensor(v, device=local_device) for k, v in self.calib_raw.items()}
        
        self.running_norm = torch.tensor(0.0, device=local_device)
        self.node_buffer = torch.zeros((self.window_size, self.hidden_dim), dtype=torch.float32, device=local_device)
        self.node_count = 0
        self.previous_flux = torch.tensor(0.0, device=local_device)
        
        self.vanguard_thresh = torch.tensor(VANGUARD_THRESHOLD_VAL, device=local_device)
        self.stab_epsilon = torch.tensor(STABILIZATION_EPSILON_VAL, device=local_device)
        self.device_initialized = True

    def set_barycentric_horizon(self, prompt_hidden_states):
        local_device = prompt_hidden_states.device
        if not self.device_initialized:
            self._initialize_device_tensors(local_device)
            
        centroid = prompt_hidden_states.mean(dim=1)[0].float()
        self.horizon_node = F.normalize(centroid, dim=0)

    def calculate_friction_and_update(self, cand_mid, cand_late):
        local_device = cand_mid.device
        if not self.device_initialized:
            self._initialize_device_tensors(local_device)
            
        v = cand_mid.float()
        norm = torch.linalg.norm(v)
        u_v = v / (norm + 1e-9)
        
        v_late = cand_late.to(local_device).float()
        u_late = v_late / (torch.linalg.norm(v_late) + 1e-9)

        if self.last_node_v is None or self.prompt_anchor is None:
            self._update_trajectory(u_v, norm)
            return torch.tensor(0.0, device=local_device)
            
        mu_depth = self.calib['mu_depth'].to(local_device)
        sigma_depth = self.calib['sigma_depth'].to(local_device)
        mu_var = self.calib['mu_var'].to(local_device)
        sigma_var = self.calib['sigma_var'].to(local_device)
        mu_flux = self.calib['mu_flux'].to(local_device)
        sigma_flux = self.calib['sigma_flux'].to(local_device)
        mu_curv = self.calib['mu_curv'].to(local_device)
        sigma_curv = self.calib['sigma_curv'].to(local_device)
        
        running_norm_local = self.running_norm.to(local_device)
        prev_flux_local = self.previous_flux.to(local_device)
        vang_thresh_local = self.vanguard_thresh.to(local_device)
        stab_eps_local = self.stab_epsilon.to(local_device)
        
        last_node_local = self.last_node_v.to(local_device)
        anchor_local = self.prompt_anchor.to(local_device)

        raw_depth = 1.0 - F.cosine_similarity(u_v, u_late, dim=0)
        z_depth = torch.clamp((raw_depth - mu_depth) / sigma_depth, min=0.0)

        raw_variance = torch.abs(norm - running_norm_local) / (running_norm_local + 1e-9)
        raw_flux = 1.0 - F.cosine_similarity(u_v, last_node_local, dim=0)
        raw_curv = 1.0 - F.cosine_similarity(u_v, anchor_local, dim=0)
        
        z_var = torch.clamp((raw_variance - mu_var) / sigma_var, min=0.0)
        z_flux = torch.clamp((raw_flux - mu_flux) / sigma_flux, min=0.0)
        z_curv = torch.clamp((raw_curv - mu_curv) / sigma_curv, min=0.0)
        
        base_z = torch.max(torch.stack([z_var, z_flux, z_curv]))
        delta_flux = torch.abs(raw_flux - prev_flux_local)
        
        if not self.guardian_active:
            if delta_flux < stab_eps_local:
                self.stable_ticks += 1
                if self.stable_ticks >= 2: self.guardian_active = True
            else:
                self.stable_ticks = 0
            if not self.guardian_active:
                self._update_trajectory(u_v, norm)
                return z_depth

        final_topo_z = base_z
        if (base_z >= vang_thresh_local or z_depth >= vang_thresh_local) and self.pivot_node is not None and self.horizon_node is not None:
            accel_penalty = delta_flux * 2.0
            t1 = F.normalize(last_node_local - self.pivot_node.to(local_device), dim=0)
            t2 = F.normalize(u_v - last_node_local, dim=0)
            torsion = 1.0 - F.cosine_similarity(t2, t1, dim=0)
            drift = 1.0 - F.cosine_similarity(u_v, self.horizon_node.to(local_device), dim=0)
            final_topo_z = base_z + accel_penalty + (torsion * 1.5) + (drift * 0.5)

        z_vol = torch.tensor(0.0, device=local_device)
        if self.node_count >= 3 and self.horizon_node is not None and self.pivot_node is not None:
            v2 = self.node_buffer[(self.node_count - 3) % self.window_size].to(local_device)
            vertices = torch.stack([
                self.horizon_node.to(local_device), 
                anchor_local, 
                v2, 
                self.pivot_node.to(local_device), 
                last_node_local, 
                u_v
            ])
            gram = torch.matmul(vertices, vertices.T)
            gram = gram + (torch.eye(6, device=local_device) * 1e-5)
            
            vol = torch.clamp(torch.abs(torch.linalg.det(gram)), min=1e-12)
            if vol < 1e-10:
                z_vol = torch.tensor(3.0, device=local_device)
            else:
                log_vol = torch.log10(vol)
                z_vol = torch.clamp((-3.0 - log_vol) * 0.5, min=0.0)

        self._update_trajectory(u_v, norm)
        return torch.max(torch.stack([final_topo_z, z_depth, z_vol]))

    def _update_trajectory(self, u_v, norm):
        local_device = u_v.device
        
        if self.horizon_node is not None:
            self.horizon_node = self.horizon_node.to(local_device)
            alpha = 0.9 if self.node_count > 5 else 0.7
            self.horizon_node = F.normalize((alpha * self.horizon_node) + ((1.0 - alpha) * u_v), dim=0)
            
            if self.node_count > 0 and self.node_count % 20 == 0 and self.prompt_anchor is not None:
                self.prompt_anchor = self.prompt_anchor.to(local_device)
                self.horizon_node = F.normalize((0.7 * self.horizon_node) + (0.3 * self.prompt_anchor), dim=0)
        else:
            self.horizon_node = u_v

        if self.last_node_v is not None:
            self.last_node_v = self.last_node_v.to(local_device)
            self.previous_flux = 1.0 - F.cosine_similarity(u_v, self.last_node_v, dim=0)
            self.pivot_node = self.last_node_v
            
        idx = self.node_count % self.window_size
        self.node_buffer[idx] = u_v
        self.node_count += 1
        
        if self.prompt_anchor is None:
            self.prompt_anchor, self.last_node_v, self.running_norm = u_v, u_v, norm
        else:
            self.last_node_v = u_v
            self.running_norm = (0.9 * self.running_norm.to(local_device)) + (0.1 * norm.to(local_device))
            valid_nodes = min(self.node_count, self.window_size)
            self.prompt_anchor = F.normalize(torch.mean(self.node_buffer[:valid_nodes], dim=0), dim=0)

# ==========================================
# 5. SOTA EPISTEMIC CONTROLLER (V41 Arbitration)
# ==========================================
class HaloEpistemicProcessor(LogitsProcessor):
    def __init__(self, twin, interceptor, exec_limit_val, raw_lm_head_normalized, truth_datastore=None):
        self.twin = twin
        self.interceptor = interceptor
        self.raw_lm_head = raw_lm_head_normalized
        self.truth_datastore = truth_datastore
        self.exec_limit_tensor = None
        self.max_entropy = None

    def __call__(self, input_ids, scores):
        with torch.no_grad():
            mid_state = self.interceptor.states.get('mid')
            late_state = self.interceptor.states.get('late')
            
            target_device = scores.device
            
            if self.exec_limit_tensor is None:
                self.exec_limit_tensor = torch.tensor(1.9, device=target_device)
                self.approx_k = min(512, scores.shape[-1])
                self.max_entropy = torch.log(torch.tensor(self.approx_k, dtype=torch.float32, device=target_device))
                
            # ---> REMOVED THE 500MB LM HEAD CACHE TRANSFER FROM HERE <---
            
            if mid_state is not None and late_state is not None:
                base_friction = self.twin.calculate_friction_and_update(mid_state[0], late_state[0]).to(target_device)
                
                if base_friction > self.exec_limit_tensor and self.twin.horizon_node is not None:
                    
                    k = min(128, scores.shape[-1])
                    top_indices = torch.topk(scores[0], k).indices
                    
                    # V41 OOM FIX: Slice the CPU tensor FIRST, then move only the 128 vectors (~2MB) to GPU
                    candidate_vecs = self.raw_lm_head[top_indices.cpu()].to(target_device)
                    
                    horizon = self.twin.horizon_node.to(target_device)
                    token_drift = 1.0 - F.cosine_similarity(candidate_vecs, horizon.unsqueeze(0), dim=-1)
                    
                    drift_mean = token_drift.mean()
                    drift_std = torch.clamp(token_drift.std(), min=1e-4)
                    adaptive_threshold = drift_mean + (1.0 * drift_std)
                    
                    top_scores_ent, _ = torch.topk(scores[0], self.approx_k)
                    probs = F.softmax(top_scores_ent - top_scores_ent.max(), dim=-1)
                    entropy = -torch.sum(probs * torch.log(probs + 1e-9), dim=-1)
                    
                    entropy_ratio = torch.clamp(entropy / self.max_entropy, max=1.0)
                    gate = torch.clamp(1.0 - entropy_ratio, min=0.1)
                    
                    dynamic_gate_threshold = 0.2 + 0.3 * (base_friction / self.exec_limit_tensor)
                    extreme_scale = 1.5 + 0.5 * (base_friction / self.exec_limit_tensor)
                    extreme_mask = token_drift > (adaptive_threshold * extreme_scale)
                    
                    effective_mask = (token_drift > adaptive_threshold) & ((gate > dynamic_gate_threshold) | extreme_mask)
                    
                    if effective_mask.sum() > (0.8 * k):
                        effective_mask[:] = False 
                    
                    if effective_mask.any():
                        token_weights = (token_drift / (token_drift.max() + 1e-6)) ** 1.5
                        
                        # V41 CROSS-MANIFOLD ARBITRATION
                        if self.truth_datastore and self.truth_datastore.is_active:
                            p_truth = self.truth_datastore.lookup(mid_state[0]).to(target_device)
                            raw_truth_support = p_truth[top_indices]
                            
                            truth_support = torch.clamp(raw_truth_support, min=1e-6)
                            truth_support = truth_support / (truth_support.sum() + 1e-9)
                            
                            max_truth = truth_support.max() + 1e-9
                            truth_discount = truth_support / max_truth
                        else:
                            truth_discount = torch.zeros(k, device=target_device)

                        pruning_indices = top_indices[effective_mask]
                        # Final penalty dynamically blends geometry against factual alignment
                        final_penalties = 20.0 * gate * token_weights[effective_mask] * (1.0 - truth_discount[effective_mask])
                        scores[0, pruning_indices] -= final_penalties
                    
        return scores

def generate_v41_answer(prompt, calibration_data, lm_head_normalized, truth_datastore=None, use_hybrid=True):
    full_prompt = f"[INST] {prompt} [/INST]"
    
    inputs = tokenizer(full_prompt, return_tensors="pt")
    inputs = {k: v.to(INPUT_DEVICE) for k, v in inputs.items()}
    
    twin = ErrorFieldTwinV40(calibration_data=calibration_data)
    interceptor = HiddenStateInterceptor(model, TARGET_LAYER)
    dynamic_exec_limit_val = calibration_data.get('dynamic_threshold', 1.9)
    
    processors = LogitsProcessorList()
    if use_hybrid:
        processors.append(HaloEpistemicProcessor(twin, interceptor, dynamic_exec_limit_val, lm_head_normalized, truth_datastore))
        
    try:
        with torch.no_grad():
            pre_flight = model(**inputs, output_hidden_states=True)
            prompt_hidden_states = pre_flight.hidden_states[TARGET_LAYER].detach()
            twin.set_barycentric_horizon(prompt_hidden_states)
            del pre_flight
            
            output_ids = model.generate(
                **inputs,
                max_new_tokens=TARGET_OUTPUT_SIZE,
                logits_processor=processors,
                use_cache=True, 
                pad_token_id=tokenizer.eos_token_id,
                do_sample=False 
            )
    finally:
        interceptor.remove() 
        
    generated_ids = output_ids[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

# ==========================================
# 6. EXECUTION & BENCHMARKING
# ==========================================
if __name__ == "__main__":
    dataset = load_dataset("truthful_qa", "generation", split="validation")
    results = []
    
    # Initialize the V41 Truth Datastore using the complete absolute paths specified at the top
    truth_db = FAISSDatastore(
        vocab_size=len(tokenizer), 
        index_path=INDEX_FILE_PATH, 
        token_map_path=TOKEN_MAP_FILE_PATH
    )
    
    for i in tqdm(range(len(dataset)), desc="HALO V41 Truth Benchmark"):
        item = dataset[i]
        q = item['question']
        
        dynamic_calib = route_and_calibrate_elastic(q)
        
        base = generate_v41_answer(q, dynamic_calib, GLOBAL_LM_HEAD_NORMALIZED, truth_datastore=truth_db, use_hybrid=False)
        steer = generate_v41_answer(q, dynamic_calib, GLOBAL_LM_HEAD_NORMALIZED, truth_datastore=truth_db, use_hybrid=True)
        
        results.append({'question': q, 'baseline': base, 'steered': steer})
        
        torch.cuda.empty_cache()
        gc.collect()

        if (i + 1) % 5 == 0:
            pd.DataFrame(results).to_csv("halo_engine_v41_output.csv", index=False)

    pd.DataFrame(results).to_csv("halo_engine_v41_output.csv", index=False)
    print(f"\n[SYSTEM SECURED] HALO Architecture V41 Processing Complete.")

In [1]:
import os
import torch
import torch.nn.functional as F
import numpy as np
import faiss
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
from tqdm import tqdm
import gc

# ==========================================
# CONFIGURATION
# ==========================================
MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.2"
TARGET_LAYER = 16
CHUNK_SIZE = 256  
MAX_TOKENS_PER_INDEX = 500000  # Will generate a ~4GB index
DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"

print(f"Initializing Tier 1 (Wikidata) Datastore Compiler on {DEVICE}...")

# 1. Load Tokenizer & Model
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, device_map="auto", torch_dtype=torch.bfloat16)
model.eval()

# ==========================================
# PHASE 1: ONTOLOGICAL CORPUS ASSEMBLY (DBpedia)
# ==========================================
def load_tier1_corpus():
    """
    Pulls structured relational facts from DBpedia 
    (Wikipedia entities converted into dense factual sentences).
    """
    print("Downloading DBpedia (Ontological Graph) dataset...")
    
    # dbpedia_14 is a rock-solid, permanently available dataset on HF
    dbpedia = load_dataset("dbpedia_14", split="train[:15000]") 
    
    corpus_texts = []
    
    print("Flattening Graph Entities into Epistemic Anchors...")
    for item in dbpedia:
        # DBpedia stores the factual summary in the 'content' field
        sentence = item['content'].strip()
        corpus_texts.append(f"Fact: {sentence}")
            
    text_blob = " ".join(corpus_texts)
    
    print("Tokenizing Ontological Corpus...")
    tokens = tokenizer(text_blob, return_tensors="pt").input_ids[0]
    print(f"Total Relational Tokens to Index: {len(tokens)}")
    
    return tokens

# ==========================================
# PHASE 2: MISTRAL VECTORIZATION
# ==========================================
def extract_hidden_states(tokens):
    total_tokens = min(len(tokens) - 1, MAX_TOKENS_PER_INDEX)
    hidden_dim = model.config.hidden_size
    
    all_hidden_states = torch.empty((total_tokens, hidden_dim), dtype=torch.float32)
    all_next_tokens = torch.empty((total_tokens,), dtype=torch.long)
    
    current_idx = 0
    
    print(f"Extracting Layer {TARGET_LAYER} Geometry...")
    for i in tqdm(range(0, total_tokens, CHUNK_SIZE)):
        end_idx = min(i + CHUNK_SIZE, total_tokens)
        chunk_input = tokens[i : end_idx + 1].unsqueeze(0).to(DEVICE)
        
        with torch.no_grad():
            outputs = model(chunk_input, output_hidden_states=True)
            layer_16_states = outputs.hidden_states[TARGET_LAYER][0] 
            
            normalized_states = F.normalize(layer_16_states[:-1], dim=-1).cpu().float()
            target_tokens = chunk_input[0, 1:].cpu()
            
            batch_size = normalized_states.shape[0]
            all_hidden_states[current_idx : current_idx + batch_size] = normalized_states
            all_next_tokens[current_idx : current_idx + batch_size] = target_tokens
            
            current_idx += batch_size
            
            del outputs, layer_16_states, normalized_states, target_tokens, chunk_input
            if i % (CHUNK_SIZE * 10) == 0:
                torch.cuda.empty_cache()
                
    return all_hidden_states[:current_idx], all_next_tokens[:current_idx]

# ==========================================
# PHASE 3: FAISS COMPILATION
# ==========================================
def build_and_save_index(hidden_states, next_tokens, index_name="tier1_wikidata"):
    print(f"\nBuilding FAISS Index: {index_name}...")
    
    hidden_dim = hidden_states.shape[1]
    hidden_np = hidden_states.numpy()
    
    index = faiss.IndexFlatIP(hidden_dim)
    
    print("Adding vectors to FAISS...")
    index.add(hidden_np)
    
    print("Saving to Kaggle Output Directory...")
    # Saving to the root Kaggle working directory
    faiss.write_index(index, f"{index_name}.index")
    torch.save(next_tokens, f"{index_name}_token_map.pt")
    
    print(f"Successfully compiled {index_name}.index and mapped {len(next_tokens)} relational tokens.")

# ==========================================
# EXECUTION
# ==========================================
if __name__ == "__main__":
    corpus_tokens = load_tier1_corpus()
    h_states, target_tokens = extract_hidden_states(corpus_tokens)
    build_and_save_index(h_states, target_tokens, index_name="tier1_wikidata")
    
    print("\n[PHASE 1 & 2 COMPLETE] Wikidata Datastore is ready for V41 Integration.")

Initializing Tier 1 (Wikidata) Datastore Compiler on cuda:0...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Flattening Graph Entities into Epistemic Anchors...
Tokenizing Ontological Corpus...
Total Relational Tokens to Index: 1204814
Extracting Layer 16 Geometry...


100%|██████████| 1954/1954 [1:08:12<00:00,  2.09s/it]



Building FAISS Index: tier1_wikidata...
Adding vectors to FAISS...
Saving to Kaggle Output Directory...
Successfully compiled tier1_wikidata.index and mapped 500000 relational tokens.

[PHASE 1 & 2 COMPLETE] Wikidata Datastore is ready for V41 Integration.


In [3]:
from IPython.display import FileLink

# Replace this with the actual name of your file or zipped folder
FileLink('tier1_wikidata.index')

/kaggle/working/tier1_wikidata.index

In [ ]:
import os
import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd
import requests
import urllib.parse
from transformers import AutoModelForCausalLM, AutoTokenizer, LogitsProcessor, LogitsProcessorList
from datasets import load_dataset
from tqdm import tqdm
import gc
import warnings

warnings.filterwarnings("ignore")

# ==========================================
# 0. CONFIGURATION & FILE PATHS
# ==========================================
MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.2"
WOLFRAM_APP_ID = "YOUR_WOLFRAM_APP_ID_HERE"  # <-- REQUIRED: Insert your key

# Kaggle Dataset Paths
WIKI_INDEX_PATH = "/kaggle/input/datasets/omrai1/faiss-data/tier1_wikidata.index"
WIKI_MAP_PATH = "/kaggle/input/datasets/omrai1/faiss-data/tier1_wikidata_token_map.pt"
FEVER_INDEX_PATH = "/kaggle/input/datasets/omrai1/faiss-data/tier2_fever.index"
FEVER_MAP_PATH = "/kaggle/input/datasets/omrai1/faiss-data/tier2_fever_token_map.pt"

TARGET_LAYER = 16          
LATE_LAYER = -1            
TARGET_OUTPUT_SIZE = 100   

VANGUARD_THRESHOLD_VAL = 1.5   
STABILIZATION_EPSILON_VAL = 0.05 

print(f"Booting HALO V41 Split-Architecture on {MODEL_ID}...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token

# Automatically splits the model across the Twin T4s
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, device_map="auto", torch_dtype=torch.bfloat16)
model.eval()

INPUT_DEVICE = next(model.parameters()).device

# Force garbage collection before doing heavy extractions
gc.collect()
torch.cuda.empty_cache()

# ==========================================
# 0.5. CHUNKED LM HEAD EXTRACTION
# ==========================================
print("Extracting LM Head weights via Chunked Identity Matrix...")
lm_head_module = model.get_output_embeddings()
hidden_dim = lm_head_module.in_features
vocab_size = lm_head_module.out_features

raw_lm_head = torch.empty((vocab_size, hidden_dim), device='cpu', dtype=torch.float32)
chunk_size = 256 

with torch.no_grad():
    zeros = torch.zeros((1, hidden_dim), device=INPUT_DEVICE, dtype=model.dtype)
    b = lm_head_module(zeros)
    
    for i in tqdm(range(0, hidden_dim, chunk_size), desc="Extracting Matrix"):
        end = min(i + chunk_size, hidden_dim)
        sz = end - i
        
        identity_chunk = torch.zeros((sz, hidden_dim), device=INPUT_DEVICE, dtype=model.dtype)
        identity_chunk[torch.arange(sz), torch.arange(i, end)] = 1.0
        
        chunk_out = lm_head_module(identity_chunk)
        w_chunk = (chunk_out - b).t().float().cpu()
        
        raw_lm_head[:, i:end] = w_chunk
        del identity_chunk, chunk_out, w_chunk
        
    del zeros, b
    torch.cuda.empty_cache()

GLOBAL_LM_HEAD_NORMALIZED = F.normalize(raw_lm_head, dim=-1)
del raw_lm_head
gc.collect()

# ==========================================
# 1. PRE-FLIGHT MACRO-TRUTH ROUTER
# ==========================================
class PreFlightTruthEngine:
    def __init__(self, wiki_idx_path, wiki_map_path, top_k=3):
        print("Loading 8GB Wikidata Index to CPU RAM for Pre-Flight RAG...")
        try:
            import faiss
            self.wiki_index = faiss.read_index(wiki_idx_path)
            self.wiki_map = torch.load(wiki_map_path)
            self.top_k = top_k
            self.is_active = True
            print("Wikidata Pre-Flight Armed.")
        except Exception as e:
            print(f"[WARNING] Wikidata index failed to load: {e}")
            self.is_active = False

    def route(self, prompt, hidden_state_query):
        prompt_lower = prompt.lower()
        context_injections = []

        # 1. Wolfram Alpha Check
        math_triggers = ['calculate', 'compute', 'equation', 'distance', 'integral', 'derivative', 'how many']
        if any(trigger in prompt_lower for trigger in math_triggers):
            encoded_query = urllib.parse.quote_plus(prompt)
            wolfram_url = f"https://www.wolframalpha.com/api/v1/llm-api?input={encoded_query}&appid={WOLFRAM_APP_ID}"
            try:
                res = requests.get(wolfram_url, timeout=5)
                if res.status_code == 200:
                    context_injections.append(f"Computational Fact: {res.text.strip()}")
            except Exception:
                pass

        # 2. Wikidata Single-Shot RAG Check
        if self.is_active:
            hidden_np = hidden_state_query.detach().cpu().float().numpy().reshape(1, -1)
            dist_w, idx_w = self.wiki_index.search(hidden_np, self.top_k)
            retrieved_tokens = self.wiki_map[idx_w[0]]
            wiki_text = tokenizer.decode(retrieved_tokens, skip_special_tokens=True).strip()
            if wiki_text:
                context_injections.append(f"Knowledge Graph Context: {wiki_text}")

        if context_injections:
            combined_context = "\n".join(context_injections)
            return f"System: Integrate these facts into your answer:\n{combined_context}\n\nUser: {prompt}"
        
        return prompt

# ==========================================
# 2. IN-FLIGHT MICRO-STEERING DATASTORE
# ==========================================
class FeverFAISSDatastore:
    def __init__(self, vocab_size, fever_idx_path, fever_map_path, k=16, temperature=0.5):
        self.vocab_size = vocab_size
        self.k = k
        self.temperature = temperature
        
        print("Loading 700MB FEVER Index to CPU RAM for In-Flight Steering...")
        try:
            import faiss
            self.index = faiss.read_index(fever_idx_path)
            self.token_map = torch.load(fever_map_path)
            self.is_active = True
            print("FEVER Datastore Armed.")
        except Exception as e:
            print(f"[WARNING] FEVER index failed to load: {e}")
            self.is_active = False

    def lookup(self, hidden_state):
        if not self.is_active:
            return torch.ones(self.vocab_size, device=hidden_state.device) / self.vocab_size
            
        hidden_np = hidden_state.detach().cpu().float().numpy().reshape(1, -1)
        distances, indices = self.index.search(hidden_np, self.k)
        weights = F.softmax(-torch.tensor(distances[0]) / self.temperature, dim=0)
        retrieved_tokens = self.token_map[indices[0]]
        
        p_truth = torch.zeros(self.vocab_size, device=hidden_state.device)
        p_truth.scatter_add_(0, retrieved_tokens.to(hidden_state.device), weights.to(hidden_state.device))
        return p_truth

# ==========================================
# 3. DYNAMIC DOMAIN CALIBRATION
# ==========================================
def route_and_calibrate_elastic(prompt):
    return {
        'mu_var': 0.1, 'sigma_var': 0.05, 'mu_flux': 0.2, 'sigma_flux': 0.05,
        'mu_curv': 0.3, 'sigma_curv': 0.08, 'mu_depth': 0.4, 'sigma_depth': 0.1,
        'dynamic_threshold': 1.9 
    }

# ==========================================
# 4. THE ROBUST INTERCEPTOR
# ==========================================
class HiddenStateInterceptor:
    def __init__(self, model, target_layer_idx):
        self.states = {}
        self.hooks = []
        
        layers_module = getattr(model.model, "layers", None)
        norm_module = getattr(model.model, "norm", getattr(model.model, "ln_f", None))
                
        def get_mid_hook():
            def hook(module, args, output):
                state = output[0] if isinstance(output, tuple) else output
                self.states['mid'] = state[:, -1, :].detach()
            return hook
            
        def get_late_hook():
            def hook(module, args, output):
                state = output[0] if isinstance(output, tuple) else output
                self.states['late'] = state[:, -1, :].detach()
            return hook

        if layers_module is not None:
            self.hooks.append(layers_module[target_layer_idx].register_forward_hook(get_mid_hook()))
        if norm_module is not None:
            self.hooks.append(norm_module.register_forward_hook(get_late_hook()))
        
    def remove(self):
        for h in self.hooks: h.remove()

# ==========================================
# 5. FULLY TENSORIZED 7-VERTEX TWIN
# ==========================================
class ErrorFieldTwinV40:
    def __init__(self, calibration_data, window_size=10, hidden_dim=4096):
        self.calib_raw = calibration_data
        self.window_size = window_size
        self.hidden_dim = hidden_dim
        self.device_initialized = False
        
        self.horizon_node = None       
        self.pivot_node = None         
        self.last_node_v = None        
        self.prompt_anchor = None      
        
        self.stable_ticks = 0
        self.guardian_active = False

    def _initialize_device_tensors(self, local_device):
        self.local_device = local_device
        self.calib = {k: torch.tensor(v, device=local_device) for k, v in self.calib_raw.items()}
        self.running_norm = torch.tensor(0.0, device=local_device)
        self.node_buffer = torch.zeros((self.window_size, self.hidden_dim), dtype=torch.float32, device=local_device)
        self.node_count = 0
        self.previous_flux = torch.tensor(0.0, device=local_device)
        self.vanguard_thresh = torch.tensor(VANGUARD_THRESHOLD_VAL, device=local_device)
        self.stab_epsilon = torch.tensor(STABILIZATION_EPSILON_VAL, device=local_device)
        self.device_initialized = True

    def set_barycentric_horizon(self, prompt_hidden_states):
        local_device = prompt_hidden_states.device
        if not self.device_initialized:
            self._initialize_device_tensors(local_device)
        centroid = prompt_hidden_states.mean(dim=1)[0].float()
        self.horizon_node = F.normalize(centroid, dim=0)

    def calculate_friction_and_update(self, cand_mid, cand_late):
        local_device = cand_mid.device
        if not self.device_initialized:
            self._initialize_device_tensors(local_device)
            
        v = cand_mid.float()
        norm = torch.linalg.norm(v)
        u_v = v / (norm + 1e-9)
        v_late = cand_late.to(local_device).float()
        u_late = v_late / (torch.linalg.norm(v_late) + 1e-9)

        if self.last_node_v is None or self.prompt_anchor is None:
            self._update_trajectory(u_v, norm)
            return torch.tensor(0.0, device=local_device)
            
        mu_depth, sigma_depth = self.calib['mu_depth'].to(local_device), self.calib['sigma_depth'].to(local_device)
        mu_var, sigma_var = self.calib['mu_var'].to(local_device), self.calib['sigma_var'].to(local_device)
        mu_flux, sigma_flux = self.calib['mu_flux'].to(local_device), self.calib['sigma_flux'].to(local_device)
        mu_curv, sigma_curv = self.calib['mu_curv'].to(local_device), self.calib['sigma_curv'].to(local_device)
        
        running_norm_local = self.running_norm.to(local_device)
        prev_flux_local = self.previous_flux.to(local_device)
        vang_thresh_local = self.vanguard_thresh.to(local_device)
        stab_eps_local = self.stab_epsilon.to(local_device)
        
        last_node_local = self.last_node_v.to(local_device)
        anchor_local = self.prompt_anchor.to(local_device)

        raw_depth = 1.0 - F.cosine_similarity(u_v, u_late, dim=0)
        z_depth = torch.clamp((raw_depth - mu_depth) / sigma_depth, min=0.0)

        raw_variance = torch.abs(norm - running_norm_local) / (running_norm_local + 1e-9)
        raw_flux = 1.0 - F.cosine_similarity(u_v, last_node_local, dim=0)
        raw_curv = 1.0 - F.cosine_similarity(u_v, anchor_local, dim=0)
        
        z_var = torch.clamp((raw_variance - mu_var) / sigma_var, min=0.0)
        z_flux = torch.clamp((raw_flux - mu_flux) / sigma_flux, min=0.0)
        z_curv = torch.clamp((raw_curv - mu_curv) / sigma_curv, min=0.0)
        
        base_z = torch.max(torch.stack([z_var, z_flux, z_curv]))
        delta_flux = torch.abs(raw_flux - prev_flux_local)
        
        if not self.guardian_active:
            if delta_flux < stab_eps_local:
                self.stable_ticks += 1
                if self.stable_ticks >= 2: self.guardian_active = True
            else:
                self.stable_ticks = 0
            if not self.guardian_active:
                self._update_trajectory(u_v, norm)
                return z_depth

        final_topo_z = base_z
        if (base_z >= vang_thresh_local or z_depth >= vang_thresh_local) and self.pivot_node is not None and self.horizon_node is not None:
            accel_penalty = delta_flux * 2.0
            t1 = F.normalize(last_node_local - self.pivot_node.to(local_device), dim=0)
            t2 = F.normalize(u_v - last_node_local, dim=0)
            torsion = 1.0 - F.cosine_similarity(t2, t1, dim=0)
            drift = 1.0 - F.cosine_similarity(u_v, self.horizon_node.to(local_device), dim=0)
            final_topo_z = base_z + accel_penalty + (torsion * 1.5) + (drift * 0.5)

        z_vol = torch.tensor(0.0, device=local_device)
        if self.node_count >= 3 and self.horizon_node is not None and self.pivot_node is not None:
            v2 = self.node_buffer[(self.node_count - 3) % self.window_size].to(local_device)
            vertices = torch.stack([
                self.horizon_node.to(local_device), anchor_local, v2, 
                self.pivot_node.to(local_device), last_node_local, u_v
            ])
            gram = torch.matmul(vertices, vertices.T) + (torch.eye(6, device=local_device) * 1e-5)
            vol = torch.clamp(torch.abs(torch.linalg.det(gram)), min=1e-12)
            if vol < 1e-10:
                z_vol = torch.tensor(3.0, device=local_device)
            else:
                log_vol = torch.log10(vol)
                z_vol = torch.clamp((-3.0 - log_vol) * 0.5, min=0.0)

        self._update_trajectory(u_v, norm)
        return torch.max(torch.stack([final_topo_z, z_depth, z_vol]))

    def _update_trajectory(self, u_v, norm):
        local_device = u_v.device
        if self.horizon_node is not None:
            self.horizon_node = self.horizon_node.to(local_device)
            alpha = 0.9 if self.node_count > 5 else 0.7
            self.horizon_node = F.normalize((alpha * self.horizon_node) + ((1.0 - alpha) * u_v), dim=0)
            if self.node_count > 0 and self.node_count % 20 == 0 and self.prompt_anchor is not None:
                self.prompt_anchor = self.prompt_anchor.to(local_device)
                self.horizon_node = F.normalize((0.7 * self.horizon_node) + (0.3 * self.prompt_anchor), dim=0)
        else:
            self.horizon_node = u_v

        if self.last_node_v is not None:
            self.last_node_v = self.last_node_v.to(local_device)
            self.previous_flux = 1.0 - F.cosine_similarity(u_v, self.last_node_v, dim=0)
            self.pivot_node = self.last_node_v
            
        idx = self.node_count % self.window_size
        self.node_buffer[idx] = u_v
        self.node_count += 1
        
        if self.prompt_anchor is None:
            self.prompt_anchor, self.last_node_v, self.running_norm = u_v, u_v, norm
        else:
            self.last_node_v = u_v
            self.running_norm = (0.9 * self.running_norm.to(local_device)) + (0.1 * norm.to(local_device))
            valid_nodes = min(self.node_count, self.window_size)
            self.prompt_anchor = F.normalize(torch.mean(self.node_buffer[:valid_nodes], dim=0), dim=0)

# ==========================================
# 6. SOTA EPISTEMIC CONTROLLER (V41 Arbitration)
# ==========================================
class HaloEpistemicProcessor(LogitsProcessor):
    def __init__(self, twin, interceptor, exec_limit_val, raw_lm_head_normalized, fever_datastore=None):
        self.twin = twin
        self.interceptor = interceptor
        self.raw_lm_head = raw_lm_head_normalized
        self.fever_datastore = fever_datastore
        self.exec_limit_tensor = None
        self.max_entropy = None

    def __call__(self, input_ids, scores):
        with torch.no_grad():
            mid_state = self.interceptor.states.get('mid')
            late_state = self.interceptor.states.get('late')
            target_device = scores.device
            
            if self.exec_limit_tensor is None:
                self.exec_limit_tensor = torch.tensor(1.9, device=target_device)
                self.approx_k = min(512, scores.shape[-1])
                self.max_entropy = torch.log(torch.tensor(self.approx_k, dtype=torch.float32, device=target_device))
            
            if mid_state is not None and late_state is not None:
                base_friction = self.twin.calculate_friction_and_update(mid_state[0], late_state[0]).to(target_device)
                
                if base_friction > self.exec_limit_tensor and self.twin.horizon_node is not None:
                    k = min(128, scores.shape[-1])
                    top_indices = torch.topk(scores[0], k).indices
                    
                    candidate_vecs = self.raw_lm_head[top_indices.cpu()].to(target_device)
                    horizon = self.twin.horizon_node.to(target_device)
                    token_drift = 1.0 - F.cosine_similarity(candidate_vecs, horizon.unsqueeze(0), dim=-1)
                    
                    drift_mean = token_drift.mean()
                    drift_std = torch.clamp(token_drift.std(), min=1e-4)
                    adaptive_threshold = drift_mean + (1.0 * drift_std)
                    
                    top_scores_ent, _ = torch.topk(scores[0], self.approx_k)
                    probs = F.softmax(top_scores_ent - top_scores_ent.max(), dim=-1)
                    entropy = -torch.sum(probs * torch.log(probs + 1e-9), dim=-1)
                    
                    entropy_ratio = torch.clamp(entropy / self.max_entropy, max=1.0)
                    gate = torch.clamp(1.0 - entropy_ratio, min=0.1)
                    
                    dynamic_gate_threshold = 0.2 + 0.3 * (base_friction / self.exec_limit_tensor)
                    extreme_scale = 1.5 + 0.5 * (base_friction / self.exec_limit_tensor)
                    extreme_mask = token_drift > (adaptive_threshold * extreme_scale)
                    
                    effective_mask = (token_drift > adaptive_threshold) & ((gate > dynamic_gate_threshold) | extreme_mask)
                    
                    if effective_mask.sum() > (0.8 * k):
                        effective_mask[:] = False 
                    
                    if effective_mask.any():
                        token_weights = (token_drift / (token_drift.max() + 1e-6)) ** 1.5
                        
                        if self.fever_datastore and self.fever_datastore.is_active:
                            p_truth = self.fever_datastore.lookup(mid_state[0]).to(target_device)
                            raw_truth_support = p_truth[top_indices]
                            
                            truth_support = torch.clamp(raw_truth_support, min=1e-6)
                            truth_support = truth_support / (truth_support.sum() + 1e-9)
                            truth_discount = truth_support / (truth_support.max() + 1e-9)
                        else:
                            truth_discount = torch.zeros(k, device=target_device)

                        pruning_indices = top_indices[effective_mask]
                        final_penalties = 20.0 * gate * token_weights[effective_mask] * (1.0 - truth_discount[effective_mask])
                        scores[0, pruning_indices] -= final_penalties
                    
        return scores

def generate_v41_answer(prompt, calibration_data, lm_head_normalized, fever_datastore=None, use_hybrid=True):
    full_prompt = f"[INST] {prompt} [/INST]"
    inputs = tokenizer(full_prompt, return_tensors="pt")
    inputs = {k: v.to(INPUT_DEVICE) for k, v in inputs.items()}
    
    twin = ErrorFieldTwinV40(calibration_data=calibration_data)
    interceptor = HiddenStateInterceptor(model, TARGET_LAYER)
    dynamic_exec_limit_val = calibration_data.get('dynamic_threshold', 1.9)
    
    processors = LogitsProcessorList()
    if use_hybrid:
        processors.append(HaloEpistemicProcessor(twin, interceptor, dynamic_exec_limit_val, lm_head_normalized, fever_datastore))
        
    try:
        with torch.no_grad():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=TARGET_OUTPUT_SIZE,
                logits_processor=processors,
                use_cache=True, 
                pad_token_id=tokenizer.eos_token_id,
                do_sample=False 
            )
    finally:
        interceptor.remove() 
        
    generated_ids = output_ids[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

# ==========================================
# 7. EXECUTION & BENCHMARKING
# ==========================================
if __name__ == "__main__":
    dataset = load_dataset("truthful_qa", "generation", split="validation")
    results = []
    
    # Initialize the split architecture
    pre_flight_engine = PreFlightTruthEngine(WIKI_INDEX_PATH, WIKI_MAP_PATH)
    fever_db = FeverFAISSDatastore(len(tokenizer), FEVER_INDEX_PATH, FEVER_MAP_PATH)
    
    for i in tqdm(range(len(dataset)), desc="HALO V41 Split-Architecture Benchmark"):
        item = dataset[i]
        original_q = item['question']
        
        # Extract hidden state for 8GB Wikidata RAG lookup
        inputs = tokenizer(original_q, return_tensors="pt").to(INPUT_DEVICE)
        with torch.no_grad():
            pre_flight_pass = model(**inputs, output_hidden_states=True)
            prompt_embed = pre_flight_pass.hidden_states[TARGET_LAYER][:, -1, :].detach()
            del pre_flight_pass
            torch.cuda.empty_cache()
            
        # 1. PRE-FLIGHT MACRO-TRUTH ROUTING (Wolfram + Wikidata)
        augmented_q = pre_flight_engine.route(original_q, prompt_embed)
        
        # 2. CALIBRATE
        dynamic_calib = route_and_calibrate_elastic(augmented_q)
        
        # Run Baseline (No steering)
        base = generate_v41_answer(augmented_q, dynamic_calib, GLOBAL_LM_HEAD_NORMALIZED, fever_datastore=None, use_hybrid=False)
        
        # Run Steered (FEVER active)
        steer = generate_v41_answer(augmented_q, dynamic_calib, GLOBAL_LM_HEAD_NORMALIZED, fever_datastore=fever_db, use_hybrid=True)
        
        results.append({'question': original_q, 'baseline': base, 'steered': steer})
        
        torch.cuda.empty_cache()
        gc.collect()

        # Save checkpoint every 5 rows
        if (i + 1) % 5 == 0:
            pd.DataFrame(results).to_csv("halo_engine_v41_split_benchmark.csv", index=False)

    pd.DataFrame(results).to_csv("halo_engine_v41_split_benchmark.csv", index=False)
    print(f"\n[SYSTEM SECURED] HALO Architecture V41 Split-Processing Complete.")